In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, '')

# B5 — Maintenance prédictive : chargement du Gold dataset

Chargement de `gold_dataset_20260622-080603.parquet` (dossier `indusense/`, en dehors de ce projet).

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

GOLD_FILE = Path('../indusense/gold_dataset_20260622-080603.parquet')

In [ ]:
from src.indusense.data import load_gold_dataset

gold = load_gold_dataset(GOLD_FILE)

print('Shape :', gold.shape)
gold.head()

In [ ]:
gold[['machine_id_std', 'window_start']].head()

- Choisir un **horizon** (ex. `label_failure_next_24h`) et construire la cible `y` (0/1).

On construit le "vecteur" des valeurs à prédire :

In [ ]:
y = gold['label_failure_next_24h'].copy()

y.value_counts(normalize=True)

In [ ]:
from src.indusense.data import drop_leaking_columns

gold, dropped_cols = drop_leaking_columns(gold)

In [ ]:
gold = gold.iloc[1:].reset_index(drop=True)
y = y.iloc[1:].reset_index(drop=True)

print('Shape :', gold.shape, '| y :', y.shape)
gold.head()

- Utiliser le **split temporel fourni** (`split_set` : train < validation < test) — **pas** de split aléatoire.

On sépare `X` (features) et `y` en train / validation / test à partir de la colonne `split_set` :

In [ ]:
from src.indusense.data import temporal_split

X = gold.drop(columns='split_set')

splits = temporal_split(gold, y, split_col='split_set')
X_train, y_train = splits['train']
X_val, y_val = splits['validation']
X_test, y_test = splits['test']

- Imputer les `NaN` (médiane) ; standardiser pour les modèles linéaires.

In [ ]:
# on liste les colonnes avec des valeurs à NaN

nan_counts = X.isna().sum()
nan_counts = nan_counts[nan_counts > 0].sort_values(ascending=False)

print(f'Colonnes avec des NaN ({len(nan_counts)}/{X.shape[1]}) :')
nan_counts

On impute donc avec la médiane :
hours_since_last_incident           4027
days_since_last_maintenance         3445
rotation_trend_6h                   1722
rotation_delta_3h                   1360
rotation_delta_1h                   1085
rotation_max_1h                      948
rotation_mean_1h                     948
rotation_std_6h                      576
rotation_mean_6h                     350
rotation_max_6h                      350
pressure_trend_6h                    115
temp_trend_6h                         99
voltage_trend_6h                      89
rotation_std_12h                      82
pressure_delta_3h                     64
temp_delta_3h                         52
voltage_delta_3h                      44
pressure_delta_1h                     30
pressure_std_6h                       22
temp_delta_1h                         20
temp_zscore_24h                       19
rotation_max_12h                      18
rotation_mean_12h                     18
temp_std_6h                           16
pressure_zscore_machine               14
voltage_delta_1h                      14
pressure_std_24h                      14
rotation_std_24h                      14
voltage_std_24h                       14
temp_std_24h                          14
voltage_std_12h                       14
pressure_std_12h                      14
temp_std_12h                          14
voltage_std_6h                        14
pressure_max_1h                       14
pressure_mean_1h                      14
temp_max_1h                            5
temp_zscore_machine                    5
temp_mean_1h                           5
pressure_max_6h                        4
pressure_mean_6h                       4

et avec 0 :
incident_max_severity_1h          133237
incident_max_severity_prev_24h    111711


In [ ]:
from src.indusense.data import impute_missing

zero_cols = ['incident_max_severity_1h', 'incident_max_severity_prev_24h']
impute_missing(X, X_train, X_val, X_test, zero_cols)

### 2. Gérer le déséquilibre
- Vérifier le taux de panne (rare) → **ne pas utiliser l'accuracy**.

In [ ]:
for name, split_y in [('train', y_train), ('validation', y_val), ('test', y_test)]:
    rate = split_y.mean()
    print(f'{name:<12} taux de panne : {rate:.2%}  ({split_y.sum()}/{len(split_y)})')

- Compenser : `class_weight="balanced"` (scikit-learn) et `scale_pos_weight` (XGBoost, calculé sur le train).

=> OK, les % sont à 16.60 (train) / 17.24 (validation) et 17.26 (test)

### 3. Entraîner trois modèles (du plus simple au plus expressif)
- **Régression logistique** — baseline linéaire (Pipeline : imputation → standardisation → modèle).

Doc : [`sklearn.linear_model.LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

feature_cols = X_train.select_dtypes(include='number').columns

log_reg = make_pipeline(
    StandardScaler(),
    LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
)
log_reg.fit(X_train[feature_cols], y_train)

- **Random Forest** — non linéaire, robuste.

Doc : [`sklearn.ensemble.RandomForestClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42,
)
rf.fit(X_train[feature_cols], y_train)

- **XGBoost** — gradient boosting, candidat principal (HP sobres, à optimiser en B7).

Doc : [`xgboost.XGBClassifier`](https://xgboost.readthedocs.io/en/stable/python/python_api.html#xgboost.XGBClassifier) — [paramètres](https://xgboost.readthedocs.io/en/stable/parameter.html)


In [ ]:
from xgboost import XGBClassifier

from src.indusense.models import compute_scale_pos_weight

scale_pos_weight = compute_scale_pos_weight(y_train)

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    eval_metric='aucpr',
    n_jobs=-1,
    random_state=42,
)
xgb.fit(X_train[feature_cols], y_train)

### 4. Évaluer et comparer
- Métriques : Choisir une métrique adaptée au problème
- **Validation croisée temporelle** (`TimeSeriesSplit`) pour une estimation robuste.
- Choisir un **seuil de décision** et lire la **matrice de confusion** (faux négatifs vs faux positifs).

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

from src.indusense.data import build_chronological_order
from src.indusense.evaluation import evaluate_models_cv

models = {
    'Régression logistique': log_reg,
    'Random Forest': rf,
    'XGBoost': xgb,
}

chrono_order = build_chronological_order(GOLD_FILE, X_train.index)
X_train_ts = X_train.loc[chrono_order]
y_train_ts = y_train.loc[chrono_order]

tscv = TimeSeriesSplit(n_splits=5)
cv_summary = evaluate_models_cv(models, X_train_ts, y_train_ts, feature_cols, tscv)
cv_summary

In [ ]:
from src.indusense.evaluation import plot_confusion_matrices

plot_confusion_matrices(models, X_val[feature_cols], y_val)

### 5. Suivre et sélectionner
- Journaliser paramètres et métriques dans **MLflow** (backend SQLite local).

In [ ]:
import mlflow

mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('b5-maintenance-predictive')



On journalise un run "baseline" par modèle : hyperparamètres tels que fixés dans les cellules de fit + métriques calculées sur la validation.

In [ ]:
from src.indusense.tracking import log_classification_run

model_params = {
    'Régression logistique': {
        'class_weight': 'balanced', 'max_iter': 1000, 'random_state': 42,
    },
    'Random Forest': {
        'n_estimators': 300, 'class_weight': 'balanced', 'n_jobs': -1, 'random_state': 42,
    },
    'XGBoost': {
        'n_estimators': 300, 'max_depth': 5, 'learning_rate': 0.1,
        'scale_pos_weight': scale_pos_weight, 'random_state': 42,
    },
}

for name, model in models.items():
    with mlflow.start_run(run_name=f'baseline_{name}'):
        log_classification_run('baseline', name, model_params[name], model, X_val[feature_cols], y_val)
        print(f'Run loggé pour {name}')

### Expérimentation : Random Forest à 100 arbres
Même config que la baseline mais `n_estimators=100` — journalisé comme run `experiment` pour comparer avec la baseline à 300 arbres.

In [ ]:
rf_100_params = {
    'n_estimators': 100, 'class_weight': 'balanced', 'n_jobs': -1, 'random_state': 42,
}
rf_100 = RandomForestClassifier(**rf_100_params)
rf_100.fit(X_train[feature_cols], y_train)

with mlflow.start_run(run_name='experiment_Random Forest 100 arbres'):
    log_classification_run('experiment', 'Random Forest (100 arbres)', rf_100_params, rf_100,
                            X_val[feature_cols], y_val)
    print('Run loggé pour Random Forest (100 arbres)')

### Expérimentation : XGBoost + GridSearchCV
Recherche sur une petite grille d'hyperparamètres, scorée en PR-AUC avec la validation croisée temporelle (`TimeSeriesSplit`), puis journalisation du meilleur modèle dans MLflow.

In [ ]:
from sklearn.model_selection import GridSearchCV

from src.indusense.tracking import log_cv_child_runs

param_grid = {
    'n_estimators': [100, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.05, 0.1],
}

scoring = {
    'pr_auc': 'average_precision',
    'roc_auc': 'roc_auc',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
}

grid_search = GridSearchCV(
    XGBClassifier(
        scale_pos_weight=scale_pos_weight,
        eval_metric='aucpr',
        n_jobs=-1,
        random_state=42,
    ),
    param_grid,
    scoring=scoring,
    refit='pr_auc',  # le "meilleur" modèle reste choisi sur la PR-AUC
    cv=tscv,  # validation croisée temporelle, sur le train ordonné chronologiquement
    n_jobs=-1,
)
grid_search.fit(X_train_ts[feature_cols], y_train_ts)

print('Meilleurs paramètres :', grid_search.best_params_)
print(f'PR-AUC moyen (CV) : {grid_search.best_score_:.4f}')

xgb_best = grid_search.best_estimator_

with mlflow.start_run(run_name='experiment_XGBoost_GridSearchCV'):
    log_classification_run(
        'experiment', 'XGBoost (GridSearchCV)',
        {**grid_search.best_params_, 'scale_pos_weight': scale_pos_weight, 'random_state': 42},
        xgb_best, X_val[feature_cols], y_val,
        extra_metrics={'cv_pr_auc': grid_search.best_score_},
    )
    log_cv_child_runs(grid_search.cv_results_, scoring, 'gridsearch_XGBoost')

### Expérimentation : XGBoost avec max_depth à 3


In [ ]:
scale_pos_weight = compute_scale_pos_weight(y_train)

xgb_md3_params = {
        'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.1,
        'scale_pos_weight': scale_pos_weight, 'random_state': 42,
    }
xgb_dw3 = XGBClassifier(**xgb_md3_params)
xgb_dw3.fit(X_train[feature_cols], y_train)

with mlflow.start_run(run_name='experiment_XGBoosta max depth 3'):
    log_classification_run('experiment', 'XGBoost max depth 3', xgb_md3_params, xgb_dw3,
                            X_val[feature_cols], y_val)
    print('Run loggé pour XGBoost max depth 3')

### Expérimentation : Régression logistique + GridSearchCV
Même démarche que pour XGBoost : grille sur la régularisation `C` (le pipeline scaler + modèle est cloné par la grille), scores multi-métriques en CV temporelle, un run enfant par combinaison.

In [ ]:
logreg_param_grid = {
    'logisticregression__C': [0.01, 0.1, 1, 10],
    'logisticregression__class_weight': ['balanced', None],
}

logreg_grid = GridSearchCV(
    make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, random_state=42),
    ),
    logreg_param_grid,
    scoring=scoring,
    refit='pr_auc',
    cv=tscv,
    n_jobs=-1,
)
logreg_grid.fit(X_train_ts[feature_cols], y_train_ts)

print('Meilleurs paramètres :', logreg_grid.best_params_)
print(f'PR-AUC moyen (CV) : {logreg_grid.best_score_:.4f}')

logreg_best = logreg_grid.best_estimator_

# on retire le préfixe "logisticregression__" ajouté par le pipeline pour la lisibilité
strip = lambda params: {k.removeprefix('logisticregression__'): v for k, v in params.items()}

with mlflow.start_run(run_name='experiment_LogReg_GridSearchCV'):
    log_classification_run(
        'experiment', 'Régression logistique (GridSearchCV)',
        {**strip(logreg_grid.best_params_), 'max_iter': 1000, 'random_state': 42},
        logreg_best, X_val[feature_cols], y_val,
        extra_metrics={'cv_pr_auc': logreg_grid.best_score_},
    )
    log_cv_child_runs(logreg_grid.cv_results_, scoring, 'gridsearch_LogReg',
                       strip_prefix='logisticregression__')

### Expérimentation : Random Forest + GridSearchCV
Même démarche : grille sur le nombre d'arbres, la profondeur et `class_weight`, scores multi-métriques en CV temporelle, un run enfant par combinaison.

In [ ]:
rf_param_grid = {
    'n_estimators': [100, 300],
    'max_depth': [None, 10, 20],
    'class_weight': ['balanced', None],
}

rf_grid = GridSearchCV(
    RandomForestClassifier(n_jobs=-1, random_state=42),
    rf_param_grid,
    scoring=scoring,
    refit='pr_auc',
    cv=tscv,
    n_jobs=-1,
)
rf_grid.fit(X_train_ts[feature_cols], y_train_ts)

print('Meilleurs paramètres :', rf_grid.best_params_)
print(f'PR-AUC moyen (CV) : {rf_grid.best_score_:.4f}')

rf_best = rf_grid.best_estimator_

with mlflow.start_run(run_name='experiment_RandomForest_GridSearchCV'):
    log_classification_run(
        'experiment', 'Random Forest (GridSearchCV)',
        {**rf_grid.best_params_, 'random_state': 42}, rf_best,
        X_val[feature_cols], y_val, extra_metrics={'cv_pr_auc': rf_grid.best_score_},
    )
    log_cv_child_runs(rf_grid.cv_results_, scoring, 'gridsearch_RandomForest')

### Expérimentation : Random Forest + Optuna
Au lieu d'une grille exhaustive, Optuna explore l'espace d'hyperparamètres de façon adaptative (TPE) : chaque essai est scoré en PR-AUC par CV temporelle, puis journalisé comme run enfant dans MLflow — même structure que les blocs GridSearchCV.

### Optuna

In [ ]:
import optuna
from sklearn.model_selection import cross_val_score

from src.indusense.tracking import log_optuna_child_runs

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 400, step=50),
        'max_depth': trial.suggest_int('max_depth', 5, 30),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.5]),
        'class_weight': trial.suggest_categorical('class_weight', ['balanced', None]),
    }
    scores = cross_val_score(
        RandomForestClassifier(**params, n_jobs=-1, random_state=42),
        X_train_ts[feature_cols], y_train_ts,
        scoring='average_precision',
        cv=tscv,
        n_jobs=-1,
    )
    return scores.mean()

sampler = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(direction='maximize', sampler=sampler)
study.optimize(objective, n_trials=25, show_progress_bar=True)

print('Meilleurs paramètres :', study.best_params)
print(f'PR-AUC moyen (CV) : {study.best_value:.4f}')

# refit du meilleur modèle sur tout le train, puis évaluation sur la validation
rf_optuna = RandomForestClassifier(**study.best_params, n_jobs=-1, random_state=42)
rf_optuna.fit(X_train[feature_cols], y_train)

with mlflow.start_run(run_name='experiment_RandomForest_Optuna'):
    log_classification_run(
        'experiment', 'Random Forest (Optuna)',
        {**study.best_params, 'random_state': 42}, rf_optuna,
        X_val[feature_cols], y_val, extra_metrics={'cv_pr_auc': study.best_value},
    )
    log_optuna_child_runs(study, 'optuna_RandomForest')

# Tableau comparatif depuis MLflow
- Relire les runs journalisés avec `mlflow.search_runs`, trier par PR-AUC et retenir le **meilleur modèle**.

In [ ]:
from src.indusense.tracking import format_run_comparison

runs = mlflow.search_runs(filter_string="tags.model_name != ''")
comparison = format_run_comparison(runs)
comparison.style.set_properties(
    subset=['Hyper paramètres'],
    **{'white-space': 'pre-wrap'}
)

# Explicabilité

### SHAP sur le meilleur modèle MLflow
On recharge depuis MLflow le run avec la meilleure PR-AUC (validation), puis on trace le `summary_plot` SHAP sur un échantillon du set de validation pour voir quelles features tirent les prédictions de panne.

In [ ]:
import shap
import mlflow

from src.indusense.explainability import build_shap_explainer, get_best_run, load_run_model, shap_failure_class

# rend la cellule autonome après un restart du kernel
mlflow.set_tracking_uri('sqlite:///mlflow.db')

# le run (parent) avec la meilleure PR-AUC sur la validation
best_run = get_best_run('b5-maintenance-predictive', "tags.model_name != ''")
print(f"Meilleur run : {best_run['tags.mlflow.runName']} "
      f"({best_run['tags.model_name']}) — PR-AUC = {best_run['metrics.pr_auc']:.4f}")

best_model = load_run_model(best_run)

# échantillon de la validation pour garder un temps de calcul raisonnable
X_shap = X_val[feature_cols].sample(n=min(1000, len(X_val)), random_state=42)

explainer, X_expl = build_shap_explainer(best_model, X_shap)

shap_values = shap_failure_class(explainer(X_expl))
shap.summary_plot(shap_values, X_expl, max_display=20)

# Explicabilité d'une prédiction de panne tirée du jeu de test

In [ ]:
# on prend dans le test une vraie panne que le modèle détecte avec la proba la plus haute
proba_test = pd.Series(
    best_model.predict_proba(X_test[feature_cols])[:, 1], index=X_test.index,
)
idx = proba_test[y_test == True].idxmax()
row = X_test.loc[[idx], feature_cols]

print(f'Ligne {idx} — vraie étiquette : panne sous 24h = {y_test.loc[idx]}')
print(f'Prédiction du modèle : proba de panne = {proba_test.loc[idx]:.3f}')

# explication SHAP de cette seule prédiction
from src.indusense.explainability import transform_for_shap

row_expl = transform_for_shap(best_model, row)
explanation = shap_failure_class(explainer(row_expl))
shap.waterfall_plot(explanation[0], max_display=15)

# On explique avec un force_plot sur le set de test

In [ ]:
# force_plot : même explication que le waterfall, vue en "tir à la corde" autour de la valeur de base
shap.force_plot(
    explanation[0].base_values,
    explanation[0].values.round(3),
    row_expl.iloc[0].round(2),      # valeurs arrondies pour des libellés lisibles
    matplotlib=True,
    figsize=(24, 4),
    text_rotation=30,               # libellés en biais pour éviter les chevauchements
    contribution_threshold=0.04,    # ne montre que les features qui pèsent vraiment
)

# Comparaison du meilleur Random Forest avec le meilleur XGBoost

In [ ]:
best_rf_run = get_best_run('b5-maintenance-predictive', "tags.model_name LIKE 'Random Forest%'")
best_xgb_run = get_best_run('b5-maintenance-predictive', "tags.model_name LIKE 'XGBoost%'")

best_rf_vs_xgb = pd.DataFrame([
    {
        'Modèle': run['tags.model_name'],
        'PR-AUC': run['metrics.pr_auc'],
        'ROC-AUC': run['metrics.roc_auc'],
        'Précision': run['metrics.precision'],
        'Rappel': run['metrics.recall'],
        'F1': run['metrics.f1'],
    }
    for run in (best_rf_run, best_xgb_run)
]).sort_values('PR-AUC', ascending=False).reset_index(drop=True)

best_rf_vs_xgb

### Comparaison des features importantes (SHAP)
On charge les deux modèles et on trace leurs `summary_plot` sur le même échantillon de validation, pour voir si Random Forest et XGBoost s'appuient sur les mêmes signaux.

In [ ]:
from src.indusense.explainability import load_best_run_model

best_rf_run, rf_model = load_best_run_model('b5-maintenance-predictive', 'Random Forest%')
best_xgb_run, xgb_model = load_best_run_model('b5-maintenance-predictive', 'XGBoost%')

# même échantillon de validation pour une comparaison équitable
X_shap_cmp = X_val[feature_cols].sample(n=min(1000, len(X_val)), random_state=42)

rf_explainer, rf_X_expl = build_shap_explainer(rf_model, X_shap_cmp)
rf_shap_values = shap_failure_class(rf_explainer(rf_X_expl))

xgb_explainer, xgb_X_expl = build_shap_explainer(xgb_model, X_shap_cmp)
xgb_shap_values = shap_failure_class(xgb_explainer(xgb_X_expl))

print(f"Random Forest ({best_rf_run['tags.model_name']}) — PR-AUC = {best_rf_run['metrics.pr_auc']:.4f}")
shap.summary_plot(rf_shap_values, rf_X_expl, max_display=15, show=False)
plt.title('Random Forest')
plt.show()

print(f"XGBoost ({best_xgb_run['tags.model_name']}) — PR-AUC = {best_xgb_run['metrics.pr_auc']:.4f}")
shap.summary_plot(xgb_shap_values, xgb_X_expl, max_display=15, show=False)
plt.title('XGBoost')
plt.show()